In [1]:
import os
os.chdir(r"C:\Users\vedik\OneDrive\Desktop\Olist Churn Project")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pandasql import sqldf

DATA = "data/"   # ← now just "data/" not "../data/"

customers  = pd.read_csv(DATA + "olist_customers_dataset.csv")
orders     = pd.read_csv(DATA + "olist_orders_dataset.csv")
items      = pd.read_csv(DATA + "olist_order_items_dataset.csv")
payments   = pd.read_csv(DATA + "olist_order_payments_dataset.csv")
reviews    = pd.read_csv(DATA + "olist_order_reviews_dataset.csv")
products   = pd.read_csv(DATA + "olist_products_dataset.csv")
sellers    = pd.read_csv(DATA + "olist_sellers_dataset.csv")
geo        = pd.read_csv(DATA + "olist_geolocation_dataset.csv")

print("✅ All files loaded!")
print(f"Orders:    {len(orders):,} rows")
print(f"Customers: {len(customers):,} rows")
print(f"Reviews:   {len(reviews):,} rows")
print(f"Items:     {len(items):,} rows")

✅ All files loaded!
Orders:    99,441 rows
Customers: 99,441 rows
Reviews:   99,224 rows
Items:     112,650 rows


In [14]:
# ============================================================
# STEP 1 — Fix date columns (they loaded as text, we need dates)
# ============================================================

# These columns look like "2017-10-02 10:56:33" but pandas
# thinks they're just text. We convert them to real dates
# so we can do math on them (e.g. how many days late?)

date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("✅ Date columns fixed")
print(orders[date_cols].dtypes)  # should now show datetime64

✅ Date columns fixed
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [13]:
# ============================================================
# STEP 2 — Keep only DELIVERED orders
# ============================================================

# Right now orders has all statuses: delivered, canceled,
# shipped, unavailable etc.
# We only want DELIVERED orders because:
# - Canceled orders never reached the customer (not a real experience)
# - Shipped orders aren't complete yet
# - We can only measure churn from completed purchases

print("Before filter:", len(orders), "orders")
print(orders['order_status'].value_counts())

orders = orders[orders['order_status'] == 'delivered']

print("\nAfter filter:", len(orders), "orders")
print("We removed:", 99441 - len(orders), "non-delivered orders")

Before filter: 96470 orders
order_status
delivered    96470
Name: count, dtype: int64

After filter: 96470 orders
We removed: 2971 non-delivered orders


In [12]:
# ============================================================
# STEP 3 — Calculate delivery delay
# ============================================================

# Delivery delay = actual delivery date MINUS estimated delivery date
# Positive number = arrived LATE (bad for customer)
# Negative number = arrived EARLY (good for customer)
# Zero = arrived exactly on time

# This is one of our most important features — we'll later
# check if late deliveries cause customers to never return

orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] - 
    orders['order_estimated_delivery_date']
).dt.days

# Quick sanity check — what does delay look like?
print("Delivery delay statistics:")
print(orders['delivery_delay_days'].describe())

print(f"\nOrders delivered LATE:  {(orders['delivery_delay_days'] > 0).sum():,}")
print(f"Orders delivered EARLY: {(orders['delivery_delay_days'] < 0).sum():,}")
print(f"Orders on time:         {(orders['delivery_delay_days'] == 0).sum():,}")
print(f"Unknown (missing date): {orders['delivery_delay_days'].isna().sum():,}")

Delivery delay statistics:
count    96470.000000
mean       -11.875889
std         10.182105
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64

Orders delivered LATE:  6,534
Orders delivered EARLY: 88,644
Orders on time:         1,292
Unknown (missing date): 0


In [11]:
# ============================================================
# STEP 4 — Drop the 8 rows with missing delivery dates
# ============================================================

# Only 8 rows out of 96,470 — too small to worry about
# Dropping them is the right call. We document why:
# "Removed 8 orders with no recorded delivery date —
#  likely a data entry issue, represents 0.008% of data"

before = len(orders)
orders = orders.dropna(subset=['order_delivered_customer_date'])
after = len(orders)

print(f"Dropped {before - after} rows with missing delivery dates")
print(f"Orders remaining: {after:,}")

Dropped 0 rows with missing delivery dates
Orders remaining: 96,470


In [10]:
# ============================================================
# STEP 5 — Build the Master Customer Table
# ============================================================

# Right now our data is spread across 8 separate files.
# A "master table" means ONE row per customer with ALL
# their information combined.
#
# Think of it like this:
# orders file     → WHEN did they buy?
# items file      → HOW MUCH did they spend?
# reviews file    → HOW HAPPY were they?
# customers file  → WHO are they and WHERE are they from?
#
# We combine all of this into one clean table.

# --- First, get average review score per order ---
# One order can have one review. We want the score.
review_scores = reviews[['order_id', 'review_score']]

# --- Join orders with reviews ---
orders_full = orders.merge(review_scores, on='order_id', how='left')

# --- Calculate total spend per order (sum of all items) ---
order_spend = items.groupby('order_id')['price'].sum().reset_index()
order_spend.columns = ['order_id', 'order_value']

# --- Join spend into orders ---
orders_full = orders_full.merge(order_spend, on='order_id', how='left')

# --- Join with customers to get customer_unique_id and state ---
# Remember: customer_unique_id is the REAL person
# customer_id changes every order — never use it for churn!
orders_full = orders_full.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_state']],
    on='customer_id', how='left'
)

print("✅ All tables joined!")
print(f"Shape: {orders_full.shape}")
print(f"Columns: {list(orders_full.columns)}")

✅ All tables joined!
Shape: (96999, 13)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_delay_days', 'review_score', 'order_value', 'customer_unique_id', 'customer_state']


In [9]:
# ============================================================
# STEP 6 — Build one row per CUSTOMER (not per order)
# ============================================================

# Right now we have one row per ORDER.
# One customer can have multiple orders.
# We need to collapse this down to one row per CUSTOMER
# with summary statistics about all their orders.
#
# This is called "aggregation" — summarizing many rows into one.

customer_df = orders_full.groupby('customer_unique_id').agg(
    total_orders        = ('order_id',                  'count'),
    total_spend         = ('order_value',                'sum'),
    avg_spend_per_order = ('order_value',                'mean'),
    avg_review_score    = ('review_score',               'mean'),
    avg_delivery_delay  = ('delivery_delay_days',        'mean'),
    first_purchase_date = ('order_purchase_timestamp',   'min'),
    last_purchase_date  = ('order_purchase_timestamp',   'max'),
    customer_state      = ('customer_state',             'first'),
).reset_index()

print("✅ Customer table built!")
print(f"Unique customers: {len(customer_df):,}")
print(f"\nFirst 3 rows:")
print(customer_df.head(3))

✅ Customer table built!
Unique customers: 93,350

First 3 rows:
                 customer_unique_id  total_orders  total_spend  \
0  0000366f3b9a7992bf8c76cfdf3221e2             1        129.9   
1  0000b849f77a49e4a4ce2b2a4ca5be3f             1         18.9   
2  0000f46a3911fa3c0805444483337064             1         69.0   

   avg_spend_per_order  avg_review_score  avg_delivery_delay  \
0                129.9               5.0                -5.0   
1                 18.9               4.0                -5.0   
2                 69.0               3.0                -2.0   

  first_purchase_date  last_purchase_date customer_state  
0 2018-05-10 10:56:27 2018-05-10 10:56:27             SP  
1 2018-05-07 11:11:27 2018-05-07 11:11:27             SP  
2 2017-03-10 21:05:03 2017-03-10 21:05:03             SC  


In [8]:
# ============================================================
# STEP 7 — Label each customer as CHURNED or ACTIVE
# ============================================================

# This is a BUSINESS DECISION we are making:
# Definition: A customer who has not ordered in 6+ months
# after their first purchase = CHURNED
#
# Why 6 months? It's a common industry standard for
# e-commerce. A customer who hasn't returned in 6 months
# is very unlikely to return at all.
#
# The dataset ends in October 2018.
# So our "reference date" = the last date in the dataset.

reference_date = orders_full['order_purchase_timestamp'].max()
print(f"Reference date (last order in dataset): {reference_date}")

# How many days since their LAST purchase?
customer_df['days_since_last_purchase'] = (
    reference_date - customer_df['last_purchase_date']
).dt.days

# How long have they been a customer? (first to last purchase)
customer_df['customer_lifespan_days'] = (
    customer_df['last_purchase_date'] - customer_df['first_purchase_date']
).dt.days

# CHURN LABEL:
# If days since last purchase > 180 (6 months) = churned (1)
# Otherwise = active (0)
customer_df['churned'] = (
    customer_df['days_since_last_purchase'] > 180
).astype(int)

# Summary
churned  = customer_df['churned'].sum()
active   = len(customer_df) - churned
churn_rate = churned / len(customer_df) * 100

print(f"\n✅ Churn labels added!")
print(f"Churned customers:  {churned:,}  ({churn_rate:.1f}%)")
print(f"Active customers:   {active:,}  ({100-churn_rate:.1f}%)")
print(f"\nOverall churn rate: {churn_rate:.1f}%")



Reference date (last order in dataset): 2018-08-29 15:00:37

✅ Churn labels added!
Churned customers:  55,004  (58.9%)
Active customers:   38,346  (41.1%)

Overall churn rate: 58.9%


In [17]:
# ============================================================
# STEP 8 — Add favourite product category per customer
# ============================================================

items_with_category = items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id', how='left'
)

items_with_category['product_category_name'] = (
    items_with_category['product_category_name'].fillna('unknown')
)

items_with_customer = items_with_category.merge(
    orders_full[['order_id', 'customer_unique_id']],
    on='order_id', how='left'
)

fav_category = (
    items_with_customer
    .groupby(['customer_unique_id', 'product_category_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .groupby('customer_unique_id')
    .first()
    .reset_index()
    [['customer_unique_id', 'product_category_name']]
    .rename(columns={'product_category_name': 'fav_category'})
)

customer_df = customer_df.merge(fav_category, on='customer_unique_id', how='left')

print("✅ Favourite category added!")
print(f"Columns now: {list(customer_df.columns)}")
print(f"\nTop 10 most common favourite categories:")
print(customer_df['fav_category'].value_counts().head(10))

✅ Favourite category added!
Columns now: ['customer_unique_id', 'total_orders', 'total_spend', 'avg_spend_per_order', 'avg_review_score', 'avg_delivery_delay', 'first_purchase_date', 'last_purchase_date', 'customer_state', 'fav_category']

Top 10 most common favourite categories:
fav_category
cama_mesa_banho           8870
beleza_saude              8446
esporte_lazer             7204
informatica_acessorios    6287
moveis_decoracao          5862
utilidades_domesticas     5393
relogios_presentes        5271
telefonia                 3939
automotivo                3744
brinquedos                3709
Name: count, dtype: int64


In [19]:
# Re-add the churned column and days_since_last_purchase
# (lost during the category merge in Step 8)

reference_date = orders_full['order_purchase_timestamp'].max()

customer_df['days_since_last_purchase'] = (
    reference_date - customer_df['last_purchase_date']
).dt.days

customer_df['customer_lifespan_days'] = (
    customer_df['last_purchase_date'] - customer_df['first_purchase_date']
).dt.days

customer_df['churned'] = (
    customer_df['days_since_last_purchase'] > 180
).astype(int)

# Now save again
customer_df.to_csv("outputs/master_customer_table.csv", index=False)

print("✅ Fixed and saved!")
print(f"Rows:     {len(customer_df):,}")
print(f"Columns:  {list(customer_df.columns)}")
print(f"Churned:  {customer_df['churned'].sum():,} ({customer_df['churned'].mean()*100:.1f}%)")

✅ Fixed and saved!
Rows:     93,350
Columns:  ['customer_unique_id', 'total_orders', 'total_spend', 'avg_spend_per_order', 'avg_review_score', 'avg_delivery_delay', 'first_purchase_date', 'last_purchase_date', 'customer_state', 'fav_category', 'days_since_last_purchase', 'customer_lifespan_days', 'churned']
Churned:  55,004 (58.9%)
